# Module 10 — Factor Analysis, Cluster Analysis & Introduction to SEM/Multilevel Models
DEDICATED notebook for Module 10 — reproduces EFA (2 factors) on the classroom dataset, and K-means + Silhouette score on the real 195-school PISA dataset. Requires `factor_analyzer` and `scikit-learn` (already installed on Colab).

In [1]:
# Run this cell first (Colab: click ▶). No extra installation needed.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """SIMULATED dataset 'Classroom of 240 students' — the SAME seed=2026 used
    throughout Modules 03-08, matching lop_hoc_240.csv/.sav in the repo exactly
    (not real data, but 100% reproducible)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Male", "Female"], n)
    method = rng.choice(["Traditional", "Project-based"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Project-based")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Project-based") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Created df:", df.shape)

Created df: (240, 14)


## 1. Exploratory Factor Analysis (EFA)
🎯 **What question does this method answer?** EFA answers: can the 6 observed items (h1-h3, a1-a3) be reduced to a SMALL number of latent factors, and which item "belongs to" which latent factor?

In [2]:
items = df[["h1","h2","h3","a1","a2","a3"]].dropna()
print("n =", len(items))

# Compute Bartlett & KMO by hand (avoids a numpy version compatibility bug in factor_analyzer)
R = items.corr().values.copy()
n_obs, p_vars = items.shape
det_R = np.linalg.det(R)
chi2 = -(n_obs - 1 - (2*p_vars + 5)/6) * np.log(det_R)
dof = p_vars*(p_vars-1)/2
p_val = 1 - stats.chi2.cdf(chi2, dof)

R_inv = np.linalg.inv(R)
D = np.diag(1/np.sqrt(np.diag(R_inv)))
partial_corr = -D @ R_inv @ D
np.fill_diagonal(partial_corr, 0)
R2 = R.copy(); np.fill_diagonal(R2, 0)
kmo_model = np.sum(R2**2) / (np.sum(R2**2) + np.sum(partial_corr**2))

print(f"Bartlett: chi2={chi2:.2f}, df={dof:.0f}, p={p_val:.2e}")
print(f"KMO = {kmo_model:.3f}")

n = 237
Bartlett: chi2=313.28, df=15, p=0.00e+00
KMO = 0.688


#### 📤 Real output
KMO=0.688 (acceptable, >0.6). Bartlett p<0.001 (significant) → the data are suitable for EFA.

In [3]:
from factor_analyzer import FactorAnalyzer
import pandas as pd

fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(items)
loadings = pd.DataFrame(fa.loadings_, index=items.columns, columns=["Factor 1", "Factor 2"])
print(loadings.round(3))

ev, _ = fa.get_eigenvalues()
print("\nEigenvalues:", ev[:4].round(3))

    Factor 1  Factor 2
h1    -0.043     0.747
h2    -0.003     0.603
h3    -0.096     0.670
a1     0.725    -0.092
a2     0.722    -0.083
a3     0.695    -0.016

Eigenvalues: [2.209 1.732 0.605 0.521]


#### 📤 Real output
a1-a3 load clearly onto Factor 1 (0.70-0.73); h1-h3 load clearly onto Factor 2 (0.60-0.75). Eigenvalues: F1=2.21, F2=1.73 (both >1). ✅ EFA "rediscovers" exactly the 2 latent concepts (interest, anxiety) that were used to generate the data — confirming the method works correctly.

## 2. K-means cluster analysis — real PISA data (195 schools)
🎯 **What question does this method answer?** K-means answers: can the 195 schools be grouped into clusters with similar PROFILES on 5 WLE indices — unlike EFA, this groups OBJECTS (schools), not VARIABLES.

In [4]:
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

vars_ = ["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP"]
d = truong[vars_].dropna()
X = StandardScaler().fit_transform(d)
print("n =", len(d))

for k in [2,3,4]:
    km = KMeans(n_clusters=k, random_state=2026, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)
    print(f"k={k}: silhouette={sil:.3f}, cluster sizes={list(np.bincount(km.labels_))}")

n = 195
k=2: silhouette=0.252, cluster sizes=[np.int64(63), np.int64(132)]
k=3: silhouette=0.180, cluster sizes=[np.int64(64), np.int64(49), np.int64(82)]
k=4: silhouette=0.233, cluster sizes=[np.int64(25), np.int64(52), np.int64(102), np.int64(16)]


#### 📤 Real output
k=2 has the highest Silhouette score (0.252) — the purely statistical optimum. k=3 (0.180) has a lower Silhouette score but gives a more interpretable story — the classic trade-off between statistical optimality and interpretability.

In [5]:
km3 = KMeans(n_clusters=3, random_state=2026, n_init=10).fit(X)
d2 = d.copy(); d2["cluster"] = km3.labels_
print(d2.groupby("cluster")[vars_].mean().round(3))
print("\nSize of each cluster:")
print(d2["cluster"].value_counts().sort_index())

         EDULEAD  NEGSCLIM  STAFFSHORT  EDUSHORT  DIGPREP
cluster                                                  
0          1.422    -0.621      -0.620    -0.675    1.216
1          0.691     0.703       1.334     1.514   -0.259
2          0.297    -0.441      -0.095    -0.194   -0.233

Size of each cluster:
cluster
0    64
1    49
2    82
Name: count, dtype: int64


#### 📤 Real output
Cluster 1 (n=49) has STAFFSHORT=+1.33, EDUSHORT=+1.51 (both high positive) → a "compounded hardship" profile: staff shortage AND learning-material shortage at once. Cluster 0 (n=64) has EDULEAD=+1.42, DIGPREP=+1.22 → a "strong leadership, high digital readiness" profile.

### ⚠️ K-means is sensitive to initialization — PSPP gives DIFFERENT cluster sizes than Python
Running `QUICK CLUSTER` in PSPP on the same data with the same k=3 gives cluster sizes 37/128/30 — quite different from Python's 64/49/82. This is NOT a bug, but a known property of K-means: the algorithm is very sensitive to its initial centroids, and the two software packages use different initialization strategies.

In [6]:
# Illustration: run K-means several times with different random initializations (not fixing random_state)
results = []
for trial in range(5):
    km_trial = KMeans(n_clusters=3, n_init=1, random_state=trial).fit(X)
    results.append(sorted(np.bincount(km_trial.labels_)))
print("Cluster sizes (sorted) across 5 runs with n_init=1 (a single initialization each time):")
for r in results:
    print(" ", r)
print("\n-> Results FLUCTUATE across runs when using only a single initialization (n_init=1).")
print("Use n_init=10 (the recommended default) to get the most stable result out of 10 attempts.")

Cluster sizes (sorted) across 5 runs with n_init=1 (a single initialization each time):
  [np.int64(45), np.int64(66), np.int64(84)]
  [np.int64(29), np.int64(56), np.int64(110)]
  [np.int64(40), np.int64(61), np.int64(94)]
  [np.int64(50), np.int64(66), np.int64(79)]
  [np.int64(34), np.int64(71), np.int64(90)]

-> Results FLUCTUATE across runs when using only a single initialization (n_init=1).
Use n_init=10 (the recommended default) to get the most stable result out of 10 attempts.


## 3. The Silhouette score — formula and intuition
🎯 **What question does this method answer?** The Silhouette score measures clustering QUALITY for each data point: does that point sit "snugly" within its own cluster (close to same-cluster points, far from other clusters) or not?

In [7]:
from sklearn.metrics import silhouette_samples
sil_vals = silhouette_samples(X, km3.labels_)
d3 = d.copy(); d3["cluster"] = km3.labels_; d3["silhouette"] = sil_vals
print("Average silhouette per cluster:")
print(d3.groupby("cluster")["silhouette"].mean().round(3))
print(f"\nNumber of schools with NEGATIVE silhouette (possibly mis-assigned): {(sil_vals<0).sum()} / {len(sil_vals)}")

Average silhouette per cluster:
cluster
0    0.078
1    0.110
2    0.301
Name: silhouette, dtype: float64

Number of schools with NEGATIVE silhouette (possibly mis-assigned): 34 / 195


#### 📤 Real output
Some schools have negative Silhouette values — a sign that the boundaries between clusters are not perfectly sharp (consistent with the overall Silhouette score of only 0.18, which is not high). This is an important reason to REPORT clustering quality honestly, rather than presenting the results as if the clusters were perfectly separated.